# Hybrid Machine Learning Framework for Predictive Fault Detection (corrected)

**Corrections compared with `hybrid_final_new.ipynb`:**

1. **Fixed random seeds** (Python, NumPy, TensorFlow, Random Forest, XGBoost, SMOTE, splits), so every run gives the same numbers.
2. **Scaling after the split.** The scaler is fitted on the training data only, then applied to the test data (no information from the test set leaks into training).
3. **Out-of-fold (OOF) stacking.** The meta-learner is trained on predictions each base model makes for samples it did *not* see during training. Before, it was trained on in-sample predictions, so it over-trusted Random Forest, which nearly memorises its training data.
4. **One consistent cross-validation.** The old Steps 20 to 22 evaluated Random Forest and XGBoost without SMOTE, and the hybrid with a different Random Forest (no depth limit) and a different neural network. Now one loop evaluates **all four models on the same folds**, with the **same preprocessing (scaling + SMOTE inside each training fold) and the same model settings** as the main experiment.
5. **Statistical tests on accuracy and F1-score**, with the hybrid compared against every base model.
6. The feature-importance plot shows feature names, and the confusion-matrix counts are printed for the paper text.

Runtime on Colab CPU: about 10 to 15 minutes (the cross-validation trains the neural network 30 times).

In [ ]:
# =========================================================
# STEP 0: INSTALL LIBRARIES (Colab already has most of these)
# =========================================================
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost imbalanced-learn tensorflow openpyxl -q

In [ ]:
# =========================================================
# STEP 1: IMPORT LIBRARIES AND FIX RANDOM SEEDS
# =========================================================
import os, random
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_rel, wilcoxon, t as t_dist

from imblearn.over_sampling import SMOTE
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print("Op determinism not available:", e)

In [ ]:
# =========================================================
# STEP 2: LOAD DATASET (Colab upload, or a local file)
# =========================================================
try:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
except ImportError:
    df = pd.read_csv("predictive_maintenance.csv")

print("Dataset Shape:", df.shape)
print(df.head())

In [ ]:
# =========================================================
# STEP 3: DATA CLEANING AND ENCODING
# =========================================================
df.drop(['UDI', 'Product ID', 'Failure Type'], axis=1, errors='ignore', inplace=True)
df = pd.get_dummies(df, columns=['Type'], drop_first=True)

target = 'Target'
X = df.drop(target, axis=1).astype(float)
y = df[target].astype(int)
feature_names = list(X.columns)
print("Features:", feature_names)
print("Class counts:", y.value_counts().to_dict())

In [ ]:
# =========================================================
# STEP 4: EDA
# =========================================================
sns.countplot(x=y)
plt.title("Class Distribution")
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(X.corr(), cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()

In [ ]:
# =========================================================
# STEP 5: TRAIN-TEST SPLIT, THEN SCALING (fit on training data only)
# =========================================================
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X.values, y.values, stratify=y.values, test_size=0.2, random_state=SEED
)

scaler = StandardScaler().fit(X_train_raw)
X_train_scaled = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# =========================================================
# STEP 6: HANDLE IMBALANCE (training data only)
# =========================================================
sm = SMOTE(sampling_strategy=0.5, random_state=SEED)
X_train, y_train = sm.fit_resample(X_train_scaled, y_train_raw)
print("Training set after SMOTE:", np.bincount(y_train))
print("Test set:", np.bincount(y_test))

In [ ]:
# =========================================================
# STEP 7: MODEL DEFINITIONS (same settings everywhere)
# =========================================================
def make_rf():
    return RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)

def make_xgb():
    return xgb.XGBClassifier(eval_metric='logloss', random_state=SEED)

def make_dl(n_features):
    model = Sequential([
        Input(shape=(n_features,)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def fit_dl(X_fit, y_fit, verbose=0):
    model = make_dl(X_fit.shape[1])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(X_fit, y_fit, epochs=50, batch_size=64, validation_split=0.2,
                        callbacks=[early_stop], verbose=verbose)
    return model, history

def train_base_models(X_fit, y_fit, verbose=0):
    rf = make_rf().fit(X_fit, y_fit)
    xgb_model = make_xgb().fit(X_fit, y_fit)
    dl_model, history = fit_dl(X_fit, y_fit, verbose=verbose)
    return (rf, xgb_model, dl_model), history

def base_probabilities(models, X_in):
    rf, xgb_model, dl_model = models
    return np.column_stack([
        rf.predict_proba(X_in)[:, 1],
        xgb_model.predict_proba(X_in)[:, 1],
        dl_model.predict(X_in, verbose=0).flatten()
    ])

In [ ]:
# =========================================================
# STEP 8: HYBRID MODEL WITH OUT-OF-FOLD STACKING
# =========================================================
def train_hybrid(X_fit, y_fit, verbose=0):
    """Returns (base models trained on all of X_fit, meta-learner, DL history).

    The meta-learner is trained on out-of-fold probabilities: for each of 5 inner
    folds, fresh base models are trained on 4/5 of the data and predict the
    remaining 1/5. The final base models are then retrained on all of X_fit.
    """
    oof = np.zeros((len(y_fit), 3))
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fit_idx, val_idx in inner.split(X_fit, y_fit):
        inner_models, _ = train_base_models(X_fit[fit_idx], y_fit[fit_idx])
        oof[val_idx] = base_probabilities(inner_models, X_fit[val_idx])

    meta_model = LogisticRegression().fit(oof, y_fit)
    base_models, history = train_base_models(X_fit, y_fit, verbose=verbose)
    return base_models, meta_model, history

base_models, meta_model, history = train_hybrid(X_train, y_train, verbose=1)
rf, xgb_model, dl_model = base_models
print("Meta-learner weights [RF, XGB, DL]:", meta_model.coef_.round(3))

In [ ]:
# =========================================================
# STEP 9: TEST PREDICTIONS AND EVALUATION (Table 1)
# =========================================================
test_probs = base_probabilities(base_models, X_test)
rf_test_prob, xgb_test_prob, dl_test_prob = test_probs.T
hybrid_prob = meta_model.predict_proba(test_probs)[:, 1]
hybrid_pred = (hybrid_prob > 0.5).astype(int)

def evaluate(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn
    }

model_probs = {
    "Random Forest": rf_test_prob,
    "XGBoost": xgb_test_prob,
    "Deep Learning": dl_test_prob,
    "Hybrid Model": hybrid_prob
}
results = pd.DataFrame({name: evaluate(y_test, p) for name, p in model_probs.items()}).T
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
results[metric_cols] = results[metric_cols].astype(float)

print("\n📊 FINAL PERFORMANCE TABLE (test set):\n")
print(results[metric_cols].round(4))
print("\nConfusion-matrix counts:\n")
print(results[["TP", "FP", "FN", "TN"]])
results.to_excel("Test_Set_Results.xlsx")

In [ ]:
# =========================================================
# STEP 10: PERFORMANCE VISUALIZATION
# =========================================================
results[metric_cols].plot(kind='bar', figsize=(10, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

# STEP 11: ROC CURVE COMPARISON (Fig. 2)
for label, p in [("RF", rf_test_prob), ("XGB", xgb_test_prob), ("DL", dl_test_prob), ("Hybrid", hybrid_prob)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=label)
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.title("ROC Curve Comparison")
plt.show()

# STEP 12: PRECISION-RECALL CURVE (Fig. 3)
precision, recall, _ = precision_recall_curve(y_test, hybrid_prob)
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

# STEP 13: CONFUSION MATRIX (Fig. 4)
cm = confusion_matrix(y_test, hybrid_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - Hybrid Model")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# STEP 14: DL TRAINING CURVE (Fig. 5)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Deep Learning Training Curve (50 Epochs)")
plt.show()

# STEP 15: FEATURE IMPORTANCE (Fig. 6)
plt.figure(figsize=(9, 5))
plt.bar(range(len(feature_names)), rf.feature_importances_)
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.ylabel("Importance")
plt.title("Feature Importance (RF)")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# STEP 16: STRATIFIED 5-FOLD CROSS-VALIDATION (all four models, same folds)
# Scaling and SMOTE are fitted inside each training fold only.
# =========================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for fold, (tr_idx, te_idx) in enumerate(cv.split(X.values, y.values), 1):
    print("=" * 70)
    print("Fold :", fold)
    print("=" * 70)

    fold_scaler = StandardScaler().fit(X.values[tr_idx])
    X_tr = fold_scaler.transform(X.values[tr_idx])
    X_te = fold_scaler.transform(X.values[te_idx])
    y_tr, y_te = y.values[tr_idx], y.values[te_idx]
    X_tr, y_tr = SMOTE(sampling_strategy=0.5, random_state=SEED).fit_resample(X_tr, y_tr)

    fold_models, fold_meta, _ = train_hybrid(X_tr, y_tr)
    P_te = base_probabilities(fold_models, X_te)
    fold_probs = {
        "Random Forest": P_te[:, 0],
        "XGBoost": P_te[:, 1],
        "Deep Learning": P_te[:, 2],
        "Hybrid Model": fold_meta.predict_proba(P_te)[:, 1]
    }
    for name, p in fold_probs.items():
        m = evaluate(y_te, p)
        cv_rows.append({"Fold": fold, "Model": name, **{k: m[k] for k in metric_cols}})
        print(f"{name:14s} " + "  ".join(f"{k}: {m[k]:.4f}" for k in metric_cols))

cv_results = pd.DataFrame(cv_rows)
cv_results.to_excel("CV_Foldwise_Results.xlsx", index=False)

In [ ]:
# =========================================================
# STEP 17: CROSS-VALIDATION SUMMARY, MEAN ± SD (Table 2)
# =========================================================
model_order = ["Random Forest", "XGBoost", "Deep Learning", "Hybrid Model"]

for name in model_order:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(cv_results[cv_results.Model == name].drop(columns="Model").round(4).to_string(index=False))

cv_mean = cv_results.groupby("Model")[metric_cols].mean().loc[model_order]
cv_sd = cv_results.groupby("Model")[metric_cols].std(ddof=1).loc[model_order]
fmt = lambda frame: frame.apply(lambda col: col.map("{:.4f}".format))
cv_summary = fmt(cv_mean) + " ± " + fmt(cv_sd)

print("\n" + "=" * 70)
print("CROSS VALIDATION SUMMARY (mean ± SD)")
print("=" * 70)
print(cv_summary.to_string())

pd.concat({"Mean": cv_mean, "SD": cv_sd}, axis=1).to_excel("CrossValidation_Summary.xlsx")

In [ ]:
# =========================================================
# STEP 18: BOXPLOT OF FOLD ACCURACY AND F1
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ["Accuracy", "F1"]):
    ax.boxplot([cv_results[cv_results.Model == m][metric].values for m in model_order],
               tick_labels=model_order)
    ax.set_title(f"Cross Validation {metric} Distribution")
    ax.set_ylabel(metric)
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# STEP 19: STATISTICAL VALIDATION (paired t-test, Wilcoxon, Cohen's d, 95% CI)
# Hybrid Model vs each base model, on fold-wise Accuracy and F1.
# Note: with only 5 folds, the Wilcoxon test cannot reach p < 0.05 (minimum p = 0.0625).
# =========================================================
def fold_scores(model, metric):
    return cv_results[cv_results.Model == model].sort_values("Fold")[metric].values

def cohens_d(a, b):
    diff = a - b
    sd = np.std(diff, ddof=1)
    return np.mean(diff) / sd if sd > 0 else np.nan

stat_rows = []
for metric in ["Accuracy", "F1"]:
    hyb = fold_scores("Hybrid Model", metric)
    for other in ["Random Forest", "XGBoost", "Deep Learning"]:
        oth = fold_scores(other, metric)
        t_val, p_t = ttest_rel(hyb, oth)
        try:
            _, p_w = wilcoxon(hyb, oth)
        except ValueError:
            p_w = np.nan
        stat_rows.append({
            "Metric": metric,
            "Comparison": f"Hybrid vs {other}",
            "Mean difference": np.mean(hyb - oth),
            "Folds Hybrid better": int((hyb > oth).sum()),
            "Paired t-value": t_val,
            "Paired t p-value": p_t,
            "Wilcoxon p-value": p_w,
            "Cohen d": cohens_d(hyb, oth)
        })

stats_table = pd.DataFrame(stat_rows)
print("=" * 70)
print("STATISTICAL VALIDATION")
print("=" * 70)
print(stats_table.round(4).to_string(index=False))
stats_table.to_excel("Statistical_Validation.xlsx", index=False)

hyb_acc = fold_scores("Hybrid Model", "Accuracy")
margin = t_dist.ppf(0.975, len(hyb_acc) - 1) * np.std(hyb_acc, ddof=1) / np.sqrt(len(hyb_acc))
print("\n95% CONFIDENCE INTERVAL (Hybrid accuracy)")
print("Mean Accuracy :", round(hyb_acc.mean(), 4))
print("95% CI :", round(hyb_acc.mean() - margin, 4), "to", round(hyb_acc.mean() + margin, 4))